# SynthFace: preview a render fix (before | after), ~10 min on a T4

Renders a few people from earlier chunks frontally, with the old look and the new code, under a warm (3000 K) and a
neutral (5500 K) key light. Nothing goes to Drive or to a chunk. Use it before rendering more chunks after a look change.

`Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

In [ ]:
SEED = 20261013
# p1_chunk02 people: afro hair (dark brown, black, grey, brown, blonde, blonde + goatee) and orange old dark skin (244, 255, 346)
IDS = "193,205,210,250,286,307,244,255,346"
DEVICE = "OPTIX"
REPO = "https://github.com/ankitscode/-synthetic-Image-Genrator.git"
BRANCH = "master"
PROJECT = "/content/synthface"
TOOLS = "/content/synthface_tools"
BLENDER = f"{TOOLS}/blender-5.2.2-linux-x64/blender"
OUT = f"{PROJECT}/output/smoke/hair_light_fix"

In [ ]:
import os, subprocess
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"
if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=True)
!git -C {PROJECT} log -1 --format="%h %cd %s" --date=short
!bash {PROJECT}/colab/setup_colab.sh {TOOLS}

In [ ]:
r = subprocess.run([BLENDER, "--background", "--python", "blender/debug_hair_light_fix.py", "--", "--seed", str(SEED),
                    "--ids", IDS, "--device", DEVICE, "--out", OUT], cwd=PROJECT, capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith("FIX_PREVIEW")))
if r.returncode != 0:
    raise RuntimeError(r.stderr[-3000:])
!cd {PROJECT} && python tools/audit_sheet.py {OUT} --tile 224
from IPython.display import Image, display
display(Image(f"{OUT}.png"))

Download the sheet (right-click > save image, or the cell below) and send it for review.

In [ ]:
from google.colab import files
files.download(f"{OUT}.png")